# Plotting Functions for Pairwise Registration

In [ ]:
# %matplotlib widget
%matplotlib inline

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import scipy as sp
import skimage as ski


from misalign.model.project import MISProjectJSON, MISProject
from misalign.model.relation import MISRelation
from misalign.model.image import HasArray
import misalign.canvas.canvas_rectangular as cr
from misalign.alignment import auto_rectangular_skimage as arski

from typing import Any
from collections.abc import Callable

In [ ]:
project_configs:list[dict]=[
    dict(mis_filepath="../example/project_a/project_a-relations-calibrated.mis.json",primary_filter=arski.Filter.simple,registration_filter=arski.Filter.rgb_gray_mean,),
    dict(mis_filepath="../example/project_b/project_b-relations-calibrated.mis.json",primary_filter=arski.Filter.simple,registration_filter=arski.Filter.rgb_gray_mean,),
    dict(mis_filepath="../example/project_c/project_c-relations-calibrated.mis.json",primary_filter=arski.Filter.simple,registration_filter=arski.Filter.rgb_gray_mean,),
    dict(mis_filepath="../example/project_d/project_d-relations-calibrated.mis.json",primary_filter=arski.Filter.simple,registration_filter=arski.Filter.rgb_gray_mean,),
    # dict(mis_filepath="../example/project_e/project_e-2-rel-cal.mis.json",primary_filter=ars.Modifier.crop(bottom=1672,filter=ars.Filter.simple),registration_filter=ars.Filter.float),),
    # dict(mis_filepath="../example/project_e/project_e-8-rel-cal.mis.json",primary_filter=ars.Modifier.crop(bottom=1672,filter=ars.Filter.simple),registration_filter=ars.Filter.float),),
    # dict(mis_filepath="../example/project_f/project_f-relations-calibrated.mis.json",primary_filter=,registration_filter=ars.Modifier.crop(bottom=4096,right=4096,filter=ars.Filter.float),),
]

In [ ]:
def pairwise_registration_project(mis_project:MISProject,relation:MISRelation,**kwargs)->arski.RectangularRegistrationResult:
    return arski.pairwise_registration(
        image_a=mis_project.get_image(image_name=relation.get_reference()[0]),
        image_b=mis_project.get_image(image_name=relation.get_reference()[1]),
        relation=relation,
        **kwargs
    )

In [ ]:
plot_axs:dict[str,list[str]]={}

def plot_image_a_image_b(
        mis_project:MISProject,relation:MISRelation,axs:dict[str,plt.Axes],
        unfiltered=False,
        formatting=True,
        **kwargs):
    image_a=mis_project.get_image(relation.get_reference()[0])
    image_b=mis_project.get_image(relation.get_reference()[1])
    if unfiltered:
        image_a=image_a.with_filter(filter=None,apply_default=False)
        image_b=image_b.with_filter(filter=None,apply_default=False)
    axs["image_a"].imshow(image_a,cmap="gray")
    axs["image_b"].imshow(image_b,cmap="gray")
    if formatting:
        axs["image_a"].set_title(relation.get_reference()[0])
        axs["image_b"].set_title(relation.get_reference()[1])
        axs["image_a"].set_axis_off()
        axs["image_b"].set_axis_off()
plot_axs[plot_image_a_image_b.__name__]=["image_a","image_b"]

# Should this actually just be a wrapper around a "render pair" function?
def plot_before_after(
        mis_project:MISProject,relation:MISRelation,axs:dict[str,plt.Axes],result:arski.RectangularRegistrationResult,
        after_offset:tuple[int,int]|None=None,before_offset:tuple[int,int]|None=None,
        focus_overlap:bool=True,focus_expand:int=50,
        formatting=True,
        **kwargs):
    
    image_a=mis_project.get_image(relation.get_reference()[0])
    image_b=mis_project.get_image(relation.get_reference()[1])

    if before_offset is None:
        before_offset:Any=relation.get_relation('r')
    if after_offset is None:
        after_offset:Any=result.optimized_offset

    renders={
        "before":cr.render_pair(image_a,image_b,offset=before_offset,weight=cr.weight_flat),
        "after":cr.render_pair(image_a,image_b,offset=after_offset,weight=cr.weight_flat)
        }
    axs["before"].imshow(renders["before"]["render"],cmap="gray")
    axs["after"].imshow(renders["after"]["render"],cmap="gray")

    if focus_overlap:
        for axs_key,offset in {"before":before_offset,"after":after_offset}.items():
            a_spans,b_spans=arski.overlap_spans(
                offset_vector=offset,
                a_shape=image_a.shape,
                b_shape=image_b.shape)
            axs[axs_key].set_xlim(
                left=renders[axs_key]["canvas_relative_offsets"][image_a.name][0]+a_spans[0][0]-focus_expand,
                right=renders[axs_key]["canvas_relative_offsets"][image_a.name][0]+a_spans[0][1]+focus_expand)
            axs[axs_key].set_ylim(
                top=renders[axs_key]["canvas_relative_offsets"][image_a.name][1]+a_spans[1][0]-focus_expand,
                bottom=renders[axs_key]["canvas_relative_offsets"][image_a.name][1]+a_spans[1][1]+focus_expand)
    if formatting:
        axs["before"].set_title(f"Before: {before_offset}")
        axs["after"].set_title(f"After: {after_offset}")
        axs["before"].set_axis_off()
        axs["after"].set_axis_off()
plot_axs[plot_before_after.__name__]=["before","after"]

def plot_local_grid(
        relation:MISRelation,axs:dict[str,plt.Axes],result:arski.RectangularRegistrationResultLocalGrid,
        overlays:bool=True,
        formatting:bool=True,
        **kwargs):

    grid_results=axs["local_grid"].imshow(result.grid_results,
        extent=(np.min(result.grid[0])-0.5,
                np.max(result.grid[0])+0.5,
                np.max(result.grid[1])+0.5,
                np.min(result.grid[1])-0.5,), #xmin,xmax,ymin,ymax
        )
    if formatting:
        axs["local_grid"].set_xlabel("Rectangular X-offset")
        axs["local_grid"].set_ylabel("Rectangular Y-offset")

        divider = make_axes_locatable(axes=axs["local_grid"])
        cax = divider.append_axes("right", size="5%", pad=0.05)
        plt.colorbar(label="Metric Results",mappable=grid_results,cax=cax)

    if overlays:
        initial_offset:Any=relation.get_relation('r')
        axs["local_grid"].scatter(
            *initial_offset,
            marker=".",
            color="r",
            label=f'Initial: {initial_offset}')

        optimized_offset:Any=result.optimized_offset
        axs["local_grid"].scatter(
            *optimized_offset,
            marker="o",
            color="r",
            label=f'Optimized: {optimized_offset}')
        axs["local_grid"].annotate("", 
            xytext=initial_offset,
            xy=optimized_offset,
            arrowprops=dict(arrowstyle="->",color="w"),)
        axs["local_grid"].legend()
        
plot_axs[plot_local_grid.__name__]=["local_grid"]

def plot_process_overlap(
        mis_project:MISProject,axs:dict[str,plt.Axes],result:arski.RectangularRegistrationResult,
        process_function:Callable[[np.ndarray,np.ndarray],np.ndarray]=np.subtract,
        filter:Callable[[HasArray],np.ndarray]=arski.Filter.float,
        **kwargs):

    array_a=filter(mis_project.get_image(relation.get_reference()[0]))
    array_b=filter(mis_project.get_image(relation.get_reference()[1]))

    overlap=arski.overlap_process(
        array_a=array_a,array_b=array_b,
        offset_ab=result.optimized_offset,process_function=process_function)

    processed=axs["process_overlap"].imshow(overlap,cmap="seismic",vmin=min(overlap.min(),-overlap.max()),vmax=max(overlap.max(),-overlap.min()))

    divider = make_axes_locatable(axes=axs["process_overlap"])
    cax = divider.append_axes("right", size="5%", pad=0.05)
    plt.colorbar(label="Process Results",mappable=processed,cax=cax)

plot_axs[plot_process_overlap.__name__]=["process_overlap"]


plt.close('all')
for project in project_configs:
    mis_project=MISProjectJSON.load(mis_filepath=project["mis_filepath"])
    mis_project.find_image_paths(mis_filepath=project["mis_filepath"])
    mis_project.set_image_filter(filter=project["primary_filter"])
    for relation in mis_project.get_relations():
        registration_result:Any=pairwise_registration_project(
            mis_project=mis_project,
            relation=relation,
            strategy=arski.StrategyLocal.local_minima_grid,
            metric=arski.LocateMetric.mean_squared_difference,
            filter=project["registration_filter"],
            strategy_max_size=20,
            strategy_footprint_shape=(9,9)
            )

        fig,axs=plt.subplot_mosaic(mosaic=[
            ["image_a"]+["image_b"],
            ["before"]+["after"],
            ["process_overlap"]+["local_grid"],
            # Could use 12 items per row to get 1/2/3/4/6 per row options > Could also use 6 to get 1/2/3 per row.
                # Could use np.lcm to get lowest common multiple and then np.repeat to convert each row into the lcm.
                # extract lists from the plotting functions `.mosaic_row` attributes.
            # 6*["image_a"]+6*["image_b"],
            # 6*["before"]+6*["after"],
            # 4*["test1"]+4*["test2"]+4*["test3"],
            # 3*["test1a"]+3*["test2a"]+3*["test3a"]+3*["test4a"]
            ],
            layout='constrained',
            )
        fig.set_figwidth(12)
        fig.set_figheight(8)
        plot_image_a_image_b(mis_project=mis_project,relation=relation,axs=axs,result=registration_result)
        plot_before_after(mis_project=mis_project,relation=relation,axs=axs,result=registration_result,)
        plot_local_grid(mis_project=mis_project,relation=relation,axs=axs,result=registration_result)
        plot_process_overlap(mis_project=mis_project,relation=relation,axs=axs,result=registration_result,filter=project["registration_filter"])
